In [ ]:
import threading
from datetime import datetime, timedelta

# ============================================================
#  SUGARBEAT – PREORDER REMINDER + ORDER PROTOTYPE
# ============================================================

# -----------------------
# OWNER SETUP
# -----------------------

print("\n=== OWNER SETUP ===")
print("Enter preorder days and system settings\n")

# Preorder Days
preorder_input = input("Enter preorder days (comma separated e.g Friday, Saturday): ").strip()
PREORDER_DAYS = [d.strip().title() for d in preorder_input.split(",") if d.strip()]
if not PREORDER_DAYS:
    PREORDER_DAYS = ["Friday"]

# Reminder Time
reminder_time = input("Reminder time (24h HH:MM, default 18:00): ").strip()
try:
    datetime.strptime(reminder_time, "%H:%M")
except:
    reminder_time = "18:00"

# -----------------------
# BUSINESS OPTIONS
# -----------------------

FLAVOR_CHOICES = {"a": "Chocolate", "b": "Vanilla", "c": "Lotus", "d": "Red Velvet"}
SIZE_CHOICES = {"a": "1 Pound", "b": "2 Pounds", "c": "3 Pounds"}
ICING_CHOICES = {"a": "Buttercream", "b": "Fresh Cream", "c": "Chocolate Ganache"}
ADDON_CHOICES = {"a": "Extra icing","b": "Sprinkles", "c": "No add-ons"}  
COLOR_CHOICES = {"a": "White", "b": "Pink", "c": "Blue", "d": "Chocolate Brown", "e": "Red"}

# -----------------------
# BASE ORDER CLASS
# -----------------------

class Order:
    def __init__(self, customer_name, quantity=1, pickup_day="Not specified"):
        self.customer_name = customer_name
        self.quantity = quantity
        self.pickup_day = pickup_day
        self.created = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def summary(self):
        print(f"Customer: {self.customer_name}")
        print(f"Quantity: {self.quantity}")
        print(f"Pickup Day: {self.pickup_day}")
        print(f"Created: {self.created}")

# -----------------------
# CAKE ORDER CLASS (INHERITANCE)
# -----------------------

class CakeOrder(Order):
    def __init__(self, customer_name, flavor, size, layers, icing, addon, color,
                 quantity=1, candles="no", message="None", pickup_day="Not specified"):
        super().__init__(customer_name, quantity, pickup_day)
        self.flavor = flavor
        self.size = size
        self.layers = layers
        self.icing = icing
        self.addon = addon
        self.color = color
        self.candles = candles
        self.message = message

    def summary(self):
        print("\n--------- ORDER SUMMARY ---------")
        super().summary()
        print(f"Flavor: {self.flavor}")
        print(f"Size: {self.size}")
        print(f"Layers: {self.layers}")
        print(f"Icing: {self.icing}")
        print(f"Color: {self.color}")
        print(f"Add-ons: {self.addon}")
        print(f"Candles: {self.candles}")
        print(f"Message: {self.message}")
        print("--------------------------------\n")

# -----------------------
# INPUT FUNCTIONS
# -----------------------

def choose_from_letter(prompt, options_dict):
    options_str = ', '.join([f"{key}) {value}" for key, value in options_dict.items()])
    full_prompt = f"{prompt} ({options_str}): "
    
    while True:
        choice = input(full_prompt).strip().lower()
        choice = choice.encode('ascii', errors='ignore').decode()
        if choice in options_dict:
            return options_dict[choice]
        print("Invalid input. Please type the letter (a, b, c...) corresponding to your choice.")

def safe_number(prompt, default=1):
    n = input(prompt).strip()
    if n.isdigit():
        return n
    return str(default)

# -----------------------
# REMINDER SYSTEM
# -----------------------

def reminder_message():
    today = datetime.now().strftime("%A")
    flavor_list = ', '.join(FLAVOR_CHOICES.values())
    preorder_list = ', '.join(PREORDER_DAYS)
    
    message = (
        f"Hello! SugarBeat is ready to make your day delicious.\n"
        f"Available flavors: {flavor_list}.\n"
        f"Preorders are open on: {preorder_list}.\n"
        f"Grab your favorite cake before it's gone!"
    )
    return message

def send_reminder():
    today = datetime.now().strftime("%A")
    if today in PREORDER_DAYS:
        print("\n=== PREORDER REMINDER ===")
        print(reminder_message())
    schedule_reminder()  # schedule next reminder

def schedule_reminder():
    now = datetime.now()
    reminder_hour, reminder_minute = map(int, reminder_time.split(":"))
    reminder_dt = now.replace(hour=reminder_hour, minute=reminder_minute, second=0, microsecond=0)
    if reminder_dt <= now:
        reminder_dt += timedelta(days=1)
    delay = (reminder_dt - now).total_seconds()
    threading.Timer(delay, send_reminder).start()

# Start first reminder
schedule_reminder()

# -----------------------
# CUSTOMER SYSTEM
# -----------------------

print("\n=== SUGARBEAT SYSTEM STARTED ===")
print("Type orders anytime. Type 'exit' to close.\n")

try:
    while True:
        name = input("Customer name (or 'exit'): ").strip()
        if name.lower() == "exit":
            break

        # Send reminder whenever a customer interacts
        print("\n=== PREORDER REMINDER ===")
        print(reminder_message())

        # Take the order
        flavor = choose_from_letter("Choose your flavor (a/b/c/d)", FLAVOR_CHOICES)
        size = choose_from_letter("Choose size (a/b/c)", SIZE_CHOICES)
        layers = safe_number("Choose the number of layers (default 1): ", 1)
        icing = choose_from_letter("Choose icing (a/b/c)", ICING_CHOICES)
        color = choose_from_letter("Choose cake color (a/b/c/d/e)", COLOR_CHOICES)
        addon = choose_from_letter("Add-ons (a/b/c)", ADDON_CHOICES)
        quantity = safe_number("Quantity (default 1): ", 1)

        candles = input("Candles (yes/no, default no): ").strip().lower()
        if candles not in ["yes", "no"]:
            candles = "no"

        message = input("Personalized message text (optional): ").strip()
        if not message:
            message = "None"

        pickup = input("Pickup day/date (in day/dd-mm-yy format): ").strip()
        if not pickup:
            pickup = "Not specified"

        # Create order and display summary
        order = CakeOrder(name, flavor, size, layers, icing, addon,
                          color, quantity, candles, message, pickup)
        order.summary()

except KeyboardInterrupt:
    pass
except Exception as e:
    print("Unexpected error:", e)

print("System closed. Goodbye.")


=== OWNER SETUP ===
Enter preorder days and system settings


=== SUGARBEAT SYSTEM STARTED ===
Type orders anytime. Type 'exit' to close.


=== PREORDER REMINDER ===
Hello! SugarBeat is ready to make your day delicious.
Available flavors: Chocolate, Vanilla, Lotus, Red Velvet.
Preorders are open on: Sunday, Tuesday.
Grab your favorite cake before it's gone!
Invalid input. Please type the letter (a, b, c...) corresponding to your choice.

--------- ORDER SUMMARY ---------
Customer: khadaija
Quantity: 3
Pickup Day: wednesday/17-12-25
Created: 2025-12-10 12:07:06
Flavor: Red Velvet
Size: 2 Pounds
Layers: 4
Icing: Buttercream
Color: Red
Add-ons: Extra icing
Candles: yes
Message: azaadi mubarak
--------------------------------

System closed. Goodbye.
